# Examen de Regresión múltiple y regularización
Dataset: wage_dataset

Entregables:
*   Notebook / script con el código y resultados.
*   Informe ejecutivo (enfocado en conclusiones para la toma de decisiones de negocio).


Una empresa de consultoría en gestión del talento humano desea entender los factores que más influyen en los salarios de los trabajadores (wage). Para esto, cuenta con el dataset wage_dataset, que incluye variables demográficas, educativas y laborales de los individuos.

El objetivo es explicar el salario a partir de las demás variables.

Se debe comparar diferentes métodos de regresión para obtener un modelo más robusto y útil para la toma de decisiones.


## Análisis exploratorio inicial:
- Describa las principales variables del dataset.
- Detecte correlaciones altas, posibles multicolinealidades y valores atípicos.

## Regresión múltiple clásica (OLS):
- Estime un modelo de regresión múltiple para explicar el salario (wage).
- Evalúe la significancia de las variables y la bondad de ajuste.

## Modelos de regresión con regularización:
- Estime un modelo con Ridge y otro con Lasso.
- Use validación cruzada para seleccionar el parámetro de penalización (α o λ).

## Comparación de desempeño:
- Compare OLS, Ridge y Lasso en términos de R², RMSE y MAE.
- Analice si los modelos con regularización mejoran el desempeño predictivo frente a la regresión clásica.

## Variables relevantes:
- Identifique las variables más influyentes en cada modelo.
- Comente si hay diferencias entre OLS y Lasso (que hace selección de variables).

## Diagnóstico de problemas:
- Evalúe si existen problemas de multicolinealidad, heterocedasticidad o sobreajuste.
- Explique cómo la regularización ayuda a mitigar algunos de estos problemas.

# Informe ejecutivo (para directivos de la empresa)
- Redacte un informe breve que incluya:


## Hallazgos principales:
- Variables que más impactan los salarios.
- Diferencias entre modelos OLS y los modelos con regularización.

## Implicaciones para la gestión del talento humano:
- ¿Qué factores deberían priorizarse para explicar diferencias salariales?
- ¿Qué riesgos existen si se ignoran problemas estadísticos como multicolinealidad?

## Recomendaciones:
- Estrategias para la política salarial.



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [133]:
import pandas as pd
path= '/content/drive/MyDrive/Colab aprendizaje supervisado/Parcial 1/wage_dataset.csv'
df= pd.read_csv(path)

In [88]:
df.head()

,age,education,experience,jobclass,region,salary
0,56,SomeCollege,9,Industrial,South,76496
1,46,SomeCollege,20,Information,West,60938
2,32,HS,19,Information,West,25206
3,60,Advanced,7,Information,East,57042
4,25,College,35,Industrial,East,27456


In [89]:
df.describe()

,age,experience,salary
count,3000.000000,3000.000000,3000.000000
mean,41.149000,19.495333,50056.518333
std,13.474989,11.462712,15196.908663
min,18.000000,0.000000,-479.000000
25%,30.000000,10.000000,39780.750000
50%,41.000000,19.000000,49860.000000
75%,53.000000,29.000000,60458.750000
max,64.000000,39.000000,101279.000000


In [134]:
# Remove rows where 'salary' is negative
df = df[df['salary'] >= 0]

print("DataFrame after removing negative salaries:")
display(df.head())

DataFrame after removing negative salaries:


,age,education,experience,jobclass,region,salary
0,56,SomeCollege,9,Industrial,South,76496
1,46,SomeCollege,20,Information,West,60938
2,32,HS,19,Information,West,25206
3,60,Advanced,7,Information,East,57042
4,25,College,35,Industrial,East,27456


In [93]:
df.describe()

,age,experience,salary
count,2999.000000,2999.000000,2999.000000
mean,41.148716,19.497833,50073.369123
std,13.477227,11.463806,15171.385395
min,18.000000,0.000000,7750.000000
25%,30.000000,10.000000,39784.500000
50%,41.000000,19.000000,49864.000000
75%,53.000000,29.000000,60463.500000
max,64.000000,39.000000,101279.000000


In [91]:
df.shape

(2999, 6)

In [92]:
df.isnull().sum()

,0
age,0
education,0
experience,0
jobclass,0
region,0
salary,0


In [135]:
# Select non-numerical columns
categorical_cols = df.select_dtypes(include='object').columns

# Display unique values and their counts for each non-numerical column
for col in categorical_cols:
    print(f"\nValue counts for column: {col}")
    display(df[col].value_counts())


Value counts for column: education


,count
education,
HS,917
College,759
SomeCollege,738
<HS,305
Advanced,280



Value counts for column: jobclass


,count
jobclass,
Information,1756
Industrial,1243



Value counts for column: region


,count
region,
West,1013
East,1011
South,975


In [136]:
from sklearn.preprocessing import OrdinalEncoder

# Define the order of education levels
education_order = ['<HS', 'HS', 'SomeCollege', 'College', 'Advanced']

# Initialize the OrdinalEncoder with the specified order
encoder = OrdinalEncoder(categories=[education_order])

# Apply ordinal encoding to the 'education' column
df['education_encoded'] = encoder.fit_transform(df[['education']])

# Display the updated DataFrame with the new encoded column
print("DataFrame with encoded education variable:")
display(df[['education', 'education_encoded']].head())

DataFrame with encoded education variable:


,education,education_encoded
0,SomeCollege,2.0
1,SomeCollege,2.0
2,HS,1.0
3,Advanced,4.0
4,College,3.0


In [137]:
# Convert non-numerical variables into dummy variables
df = pd.get_dummies(df, columns=['jobclass', 'region'], drop_first=False)

# Convert boolean columns to integers (0 or 1)
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

print("DataFrame with dummy variables:")
display(df.head())

DataFrame with dummy variables:


,age,education,experience,salary,education_encoded,jobclass_Industrial,jobclass_Information,region_East,region_South,region_West
0,56,SomeCollege,9,76496,2.0,1,0,0,1,0
1,46,SomeCollege,20,60938,2.0,0,1,0,0,1
2,32,HS,19,25206,1.0,0,1,0,0,1
3,60,Advanced,7,57042,4.0,0,1,1,0,0
4,25,College,35,27456,3.0,1,0,1,0,0


In [138]:
# Drop the original 'education' column
df = df.drop('education', axis=1)

print("DataFrame after dropping original 'education' column:")
display(df.head())

DataFrame after dropping original 'education' column:


,age,experience,salary,education_encoded,jobclass_Industrial,jobclass_Information,region_East,region_South,region_West
0,56,9,76496,2.0,1,0,0,1,0
1,46,20,60938,2.0,0,1,0,0,1
2,32,19,25206,1.0,0,1,0,0,1
3,60,7,57042,4.0,0,1,1,0,0
4,25,35,27456,3.0,1,0,1,0,0


## Partición entrenamiento y prueba

In [139]:
train= df.sample(frac=0.7, random_state=42)
test= df.drop(train.index)

In [140]:
train.head()

,age,experience,salary,education_encoded,jobclass_Industrial,jobclass_Information,region_East,region_South,region_West
1377,60,25,46360,2.0,1,0,0,0,1
932,51,27,45791,3.0,1,0,0,0,1
144,59,28,46822,1.0,1,0,0,1,0
1753,56,22,43619,4.0,0,1,0,0,1
51,64,30,73822,2.0,1,0,0,1,0


In [141]:
from sklearn.model_selection import train_test_split

X = df.drop('salary', axis=1)
y = df['salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [142]:
df_train = pd.concat([y_train, X_train], axis=1)
df_train.head()

,salary,age,experience,education_encoded,jobclass_Industrial,jobclass_Information,region_East,region_South,region_West
858,58414,26,35,1.0,1,0,0,1,0
1011,76337,41,26,0.0,1,0,1,0,0
48,30088,19,39,3.0,1,0,0,0,1
719,74120,22,20,3.0,0,1,0,0,1
1488,43121,61,7,3.0,0,1,0,0,1


In [143]:
corr_matrix = df_train.corr()
corr_matrix

,salary,age,experience,education_encoded,jobclass_Industrial,jobclass_Information,region_East,region_South,region_West
salary,1.000000,-0.043008,0.024051,-0.001138,0.002459,-0.002459,-0.003041,-0.016812,0.019701
age,-0.043008,1.000000,0.025612,-0.011182,0.032750,-0.032750,0.034488,-0.025291,-0.009425
experience,0.024051,0.025612,1.000000,0.002224,0.032244,-0.032244,-0.010066,0.059100,-0.048501
education_encoded,-0.001138,-0.011182,0.002224,1.000000,-0.012336,0.012336,0.050235,-0.014106,-0.036256
jobclass_Industrial,0.002459,0.032750,0.032244,-0.012336,1.000000,-1.000000,0.045420,0.015649,-0.060928
jobclass_Information,-0.002459,-0.032750,-0.032244,0.012336,-1.000000,1.000000,-0.045420,-0.015649,0.060928
region_East,-0.003041,0.034488,-0.010066,0.050235,0.045420,-0.045420,1.000000,-0.495486,-0.508986
region_South,-0.016812,-0.025291,0.059100,-0.014106,0.015649,-0.015649,-0.495486,1.000000,-0.495486
region_West,0.019701,-0.009425,-0.048501,-0.036256,-0.060928,0.060928,-0.508986,-0.495486,1.000000


## Modelos OLS

### Modelo original

In [144]:
import statsmodels.api as sm

# Intercepto
X_const = sm.add_constant(X_train)

# Modelo de regresión
modelo_sm = sm.OLS(y_train, X_const).fit()

# Imprimir el resumen del modelo (incluye la ecuación y ANOVA)
print(modelo_sm.summary())

# Obtener y mostrar los residuales
residuales = modelo_sm.resid
print("\nResiduales:")
print(residuales)

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.062
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.383
Time:                        15:34:00   Log-Likelihood:                -23175.
No. Observations:                2099   AIC:                         4.636e+04
Df Residuals:                    2092   BIC:                         4.640e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 2.797e+04 

In [147]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Calculate predictions
X_test_const = sm.add_constant(X_test)
predictions = modelo_sm.predict(X_test_const)

# Calculate metrics
r2 = modelo_sm.rsquared
adj_r2 = modelo_sm.rsquared_adj
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
# Avoid division by zero in MAPE
mape = np.mean(np.abs((y_test - predictions) / y_test)) * 100

print(f"R2: {r2:.4f}")
print(f"Adjusted R2: {adj_r2:.4f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"MAPE: {mape:.2f}%")

R2: 0.0030
Adjusted R2: 0.0002
RMSE: 15271.20
MAE: 12233.81
MAPE: 30.64%


### Modelo sin región

In [148]:
import statsmodels.api as sm

# Drop region columns from the training data
X_train_no_region = X_train.drop(['region_East', 'region_South', 'region_West'], axis=1)

# Add constant
X_train_no_region_const = sm.add_constant(X_train_no_region)

# Fit OLS model without region
modelo_sm_no_region = sm.OLS(y_train, X_train_no_region_const).fit()

# Print model summary
print(modelo_sm_no_region.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.309
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.264
Time:                        15:34:56   Log-Likelihood:                -23175.
No. Observations:                2099   AIC:                         4.636e+04
Df Residuals:                    2094   BIC:                         4.639e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 3.422e+04 

### Modelo Backward

In [149]:
def backward_elimination(X_train, y_train):

    selected_features = list(X_train.columns)
    X_temp_initial = sm.add_constant(X_train[selected_features])
    model_initial = sm.OLS(y_train, X_temp_initial).fit()
    best_aic = model_initial.aic
    print(f"AIC inicial with all features: {best_aic:.2f}")

    while selected_features:
        feature_to_remove = None
        improved = False
        worst_aic = best_aic # Inicializar con el mejor AIC

        for feature in selected_features:
            temp_features = [f for f in selected_features if f != feature]
            if not temp_features:
                 X_temp = pd.DataFrame({'const': np.ones(len(X_train))})
            else:
                X_temp = sm.add_constant(X_train[temp_features])

            model = sm.OLS(y_train, X_temp).fit()
            current_aic = model.aic

            # En backward elimination, se eliminan las variables que no están, cuando se obtuvo el mejor AIC
            if current_aic < best_aic:
                best_aic = current_aic
                feature_to_remove = feature
                improved = True

        if improved:
            selected_features.remove(feature_to_remove)
            print(f"Variables removidas {feature_to_remove}. Nuevo mejor AIC: {best_aic:.2f}")
        else:
            break

    return selected_features

selected_features_backward = backward_elimination(X_train, y_train)
print("\nVariables seleccionadas usando backward elimination:", selected_features_backward)

AIC inicial with all features: 46363.59
Variables removidas education_encoded. Nuevo mejor AIC: 46361.59
Variables removidas experience. Nuevo mejor AIC: 46361.06

Variables seleccionadas usando backward elimination: ['age', 'jobclass_Industrial', 'jobclass_Information', 'region_East', 'region_South', 'region_West']


In [150]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

# Add a constant to the selected features for the test set
X_test_selected = sm.add_constant(X_test[selected_features_backward])

# Re-train the model with the selected features from backward elimination
X_train_selected = sm.add_constant(X_train[selected_features_backward])
modelo_sm_selected = sm.OLS(y_train, X_train_selected).fit()


# Make predictions on the test set using the newly trained model
y_pred_backward = modelo_sm_selected.predict(X_test_selected)

# Calculate performance metrics
rmse_backward = np.sqrt(mean_squared_error(y_test, y_pred_backward))
mae_backward = mean_absolute_error(y_test, y_pred_backward)
r2_backward = r2_score(y_test, y_pred_backward)

print("Performance Metrics for Backward Elimination Model:")
print(f"R-squared (R²): {r2_backward:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_backward:.2f}")
print(f"Mean Absolute Error (MAE): {mae_backward:.2f}")
print(f"AIC: {modelo_sm_selected.aic:.2f}")
print(f"BIC: {modelo_sm_selected.bic:.2f}")

Performance Metrics for Backward Elimination Model:
R-squared (R²): 0.0015
Root Mean Squared Error (RMSE): 15274.06
Mean Absolute Error (MAE): 12221.75
AIC: 46361.06
BIC: 46389.31


### Modelo sin age

In [164]:
import statsmodels.api as sm

# Drop 'age' column from the training data
X_train_no_age = X_train.drop('age', axis=1)

# Add constant
X_train_no_age_const = sm.add_constant(X_train_no_age)

# Fit OLS model without age
modelo_sm_no_age = sm.OLS(y_train, X_train_no_age_const).fit()

# Print model summary
print(modelo_sm_no_age.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.4637
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.803
Time:                        15:53:23   Log-Likelihood:                -23177.
No. Observations:                2099   AIC:                         4.637e+04
Df Residuals:                    2093   BIC:                         4.640e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 2.687e+04 

### Modelo sin experience

In [165]:
import statsmodels.api as sm

# Drop 'experience' column from the training data
X_train_no_experience = X_train.drop('experience', axis=1)

# Add constant
X_train_no_experience_const = sm.add_constant(X_train_no_experience)

# Fit OLS model without experience
modelo_sm_no_experience = sm.OLS(y_train, X_train_no_experience_const).fit()

# Print model summary
print(modelo_sm_no_experience.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.9817
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.427
Time:                        15:54:54   Log-Likelihood:                -23176.
No. Observations:                2099   AIC:                         4.636e+04
Df Residuals:                    2093   BIC:                         4.640e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 2.833e+04 

### Modelo sin age, experience y education_encoded

In [166]:
import statsmodels.api as sm

# Drop 'age', 'experience', and 'education_encoded' columns from the training data
X_train_reduced = X_train.drop(['age', 'experience', 'education_encoded'], axis=1)

# Add constant
X_train_reduced_const = sm.add_constant(X_train_reduced)

# Fit OLS model with reduced features
modelo_sm_reduced = sm.OLS(y_train, X_train_reduced_const).fit()

# Print model summary
print(modelo_sm_reduced.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                      -0.004
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                    -1.985
Date:                Thu, 18 Sep 2025   Prob (F-statistic):               1.00
Time:                        15:56:06   Log-Likelihood:                -23182.
No. Observations:                2099   AIC:                         4.637e+04
Df Residuals:                    2094   BIC:                         4.640e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                -1.701e+17 

### Modelo sin region y jobclass

In [168]:
import statsmodels.api as sm

# Drop region and jobclass columns from the training data
X_train_no_region_jobclass = X_train.drop(['region_East', 'region_South', 'region_West', 'jobclass_Industrial', 'jobclass_Information'], axis=1)

# Add constant
X_train_no_region_jobclass_const = sm.add_constant(X_train_no_region_jobclass)

# Fit OLS model without region and jobclass
modelo_sm_no_region_jobclass = sm.OLS(y_train, X_train_no_region_jobclass_const).fit()

# Print model summary
print(modelo_sm_no_region_jobclass.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.740
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.157
Time:                        15:58:02   Log-Likelihood:                -23175.
No. Observations:                2099   AIC:                         4.636e+04
Df Residuals:                    2095   BIC:                         4.638e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const              5.131e+04   1318.83

### Modelo sin jobclass_information y region

In [169]:
import statsmodels.api as sm

# Drop 'jobclass_Information' and region columns from the training data
X_train_reduced_2 = X_train.drop(['jobclass_Information', 'region_East', 'region_South', 'region_West'], axis=1)

# Add constant
X_train_reduced_const_2 = sm.add_constant(X_train_reduced_2)

# Fit OLS model with reduced features
modelo_sm_reduced_2 = sm.OLS(y_train, X_train_reduced_const_2).fit()

# Print model summary
print(modelo_sm_reduced_2.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.309
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.264
Time:                        15:59:30   Log-Likelihood:                -23175.
No. Observations:                2099   AIC:                         4.636e+04
Df Residuals:                    2094   BIC:                         4.639e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                5.128e+04   1

### Modelo sin jobclass_industrial y region_south

In [177]:
import statsmodels.api as sm

# Drop 'jobclass_Information' and 'region_West' columns from the training data
X_train_reduced_3 = X_train.drop(['jobclass_Industrial', 'region_South'], axis=1)

# Add constant
X_train_reduced_const_3 = sm.add_constant(X_train_reduced_3)

# Fit OLS model with reduced features
modelo_sm_reduced_3 = sm.OLS(y_train, X_train_reduced_const_3).fit()

# Print model summary
print(modelo_sm_reduced_3.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.062
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.383
Time:                        16:03:37   Log-Likelihood:                -23175.
No. Observations:                2099   AIC:                         4.636e+04
Df Residuals:                    2092   BIC:                         4.640e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 5.093e+04 

### Modelo sin jobclass_information y region_east

In [183]:
import statsmodels.api as sm

# Drop 'jobclass_Information' and 'region_East' columns from the training data
X_train_reduced_4 = X_train.drop(['jobclass_Information', 'region_East'], axis=1)

# Add constant
X_train_reduced_const_4 = sm.add_constant(X_train_reduced_4)

# Fit OLS model with reduced features
modelo_sm_reduced_4 = sm.OLS(y_train, X_train_reduced_const_4).fit()

# Print model summary
print(modelo_sm_reduced_4.summary())

                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.062
Date:                Thu, 18 Sep 2025   Prob (F-statistic):              0.383
Time:                        16:18:47   Log-Likelihood:                -23175.
No. Observations:                2099   AIC:                         4.636e+04
Df Residuals:                    2092   BIC:                         4.640e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                 5.12e+04   1

## Regresión con regularización

In [178]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Define the features (X) and target (y)
# Using all features from X_train and X_test as SFS is not defined
X_train_regularized = X_train
X_test_regularized = X_test

# Add a small epsilon to prevent division by zero in MAPE
epsilon = 1e-10

# Ridge Regression
ridge_model = Ridge(alpha=1.0) # You can tune alpha
ridge_model.fit(X_train_regularized, y_train)
y_pred_ridge = ridge_model.predict(X_test_regularized)

print("--- Ridge Regression ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_ridge):.2f}")
mape_ridge = np.mean(np.abs((y_test - y_pred_ridge) / (y_test + epsilon))) * 100
print(f"MAPE: {mape_ridge:.2f}%")
print(f"R2: {r2_score(y_test, y_pred_ridge):.4f}")

# Lasso Regression
lasso_model = Lasso(alpha=1.0) # You can tune alpha
lasso_model.fit(X_train_regularized, y_train)
y_pred_lasso = lasso_model.predict(X_test_regularized)

print("\n--- Lasso Regression ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lasso):.2f}")
mape_lasso = np.mean(np.abs((y_test - y_pred_lasso) / (y_test + epsilon))) * 100
print(f"MAPE: {mape_lasso:.2f}%")
print(f"R2: {r2_score(y_test, y_pred_lasso):.4f}")

--- Ridge Regression ---
RMSE: 15271.22
MAE: 12233.83
MAPE: 30.64%
R2: 0.0019

--- Lasso Regression ---
RMSE: 15271.46
MAE: 12234.04
MAPE: 30.64%
R2: 0.0018


In [179]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Assuming X_train_regularized, X_test_regularized, y_train, y_test, and epsilon are defined in previous cells

# Define the parameter grid for alpha
param_grid = {'alpha': np.logspace(-4, 4, 100)} # Explore a wide range of alpha values

# Ridge Regression with Cross-Validation
ridge_cv = GridSearchCV(Ridge(), param_grid, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(X_train_regularized, y_train)

best_alpha_ridge = ridge_cv.best_params_['alpha']
print(f"Best alpha for Ridge: {best_alpha_ridge:.4f}")

# Train Ridge model with the best alpha
best_ridge_model = Ridge(alpha=best_alpha_ridge)
best_ridge_model.fit(X_train_regularized, y_train)
y_pred_ridge_cv = best_ridge_model.predict(X_test_regularized)

# Calculate metrics for the best Ridge model
rmse_ridge_cv = np.sqrt(mean_squared_error(y_test, y_pred_ridge_cv))
mae_ridge_cv = mean_absolute_error(y_test, y_pred_ridge_cv)
mape_ridge_cv = np.mean(np.abs((y_test - y_pred_ridge_cv) / (y_test + epsilon))) * 100
r2_ridge_cv = r2_score(y_test, y_pred_ridge_cv)


print("\n--- Ridge Regression with Cross-Validation ---")
print(f"RMSE: {rmse_ridge_cv:.2f}")
print(f"MAE: {mae_ridge_cv:.2f}")
print(f"MAPE: {mape_ridge_cv:.2f}%")
print(f"R2: {r2_ridge_cv:.4f}")


# Lasso Regression with Cross-Validation
lasso_cv = GridSearchCV(Lasso(), param_grid, scoring='neg_mean_squared_error', cv=5)
lasso_cv.fit(X_train_regularized, y_train)

best_alpha_lasso = lasso_cv.best_params_['alpha']
print(f"\nBest alpha for Lasso: {best_alpha_lasso:.4f}")

# Train Lasso model with the best alpha
best_lasso_model = Lasso(alpha=best_alpha_lasso)
best_lasso_model.fit(X_train_regularized, y_train)
y_pred_lasso_cv = best_lasso_model.predict(X_test_regularized)

# Calculate metrics for the best Lasso model
rmse_lasso_cv = np.sqrt(mean_squared_error(y_test, y_pred_lasso_cv))
mae_lasso_cv = mean_absolute_error(y_test, y_pred_lasso_cv)
mape_lasso_cv = np.mean(np.abs((y_test - y_pred_lasso_cv) / (y_test + epsilon))) * 100
r2_lasso_cv = r2_score(y_test, y_pred_lasso_cv)


print("\n--- Lasso Regression with Cross-Validation ---")
print(f"RMSE: {rmse_lasso_cv:.2f}")
print(f"MAE: {mae_lasso_cv:.2f}")
print(f"MAPE: {mape_lasso_cv:.2f}%")
print(f"R2: {r2_lasso_cv:.4f}")

Best alpha for Ridge: 10000.0000

--- Ridge Regression with Cross-Validation ---
RMSE: 15291.19
MAE: 12245.78
MAPE: 30.69%
R2: -0.0008


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.215e+10, tolerance: 3.885e+07
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.215e+10, tolerance: 3.885e+07
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.215e+10, tolerance: 3.885e


Best alpha for Lasso: 291.5053

--- Lasso Regression with Cross-Validation ---
RMSE: 15292.98
MAE: 12246.52
MAPE: 30.69%
R2: -0.0010


## Comparaciones

In [184]:
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Function to calculate AIC and BIC for scikit-learn models (approximation)
def calculate_aic_bic(model, X_train, y_train):
    n = X_train.shape[0]
    # For linear models, parameters include coefficients and intercept
    k = len(model.coef_) + 1 if model.fit_intercept else len(model.coef_)

    y_pred_train = model.predict(X_train)
    rss = np.sum((y_train - y_pred_train)**2)

    # Calculate log-likelihood (approximation for linear models with normally distributed errors)
    # L = -n/2 * log(2*pi*rss/n) - n/2
    # AIC = -2*L + 2*k
    # BIC = -2*L + k*log(n)

    # Simplified AIC and BIC based on RSS
    aic = n * np.log(rss / n) + 2 * k
    bic = n * np.log(rss / n) + k * np.log(n)

    return aic, bic


print("--- Model Performance Summary ---")

# Metrics for the first OLS model (All Features) - from cell yO5OJKABcR8W
print("\nLinear Regression (OLS - All Features):")
print(f"R2: {r2:.4f}")
print(f"Adjusted R2: {adj_r2:.4f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"AIC: {modelo_sm.aic:.2f}") # AIC from statsmodels OLS
print(f"BIC: {modelo_sm.bic:.2f}") # BIC from statsmodels OLS


# Metrics for Backward Elimination Model - from cell aj84idwSWD4z
print("\nBackward Elimination Model:")
print(f"R-squared (R²): {r2_backward:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_backward:.2f}")
print(f"Mean Absolute Error (MAE): {mae_backward:.2f}")
print(f"AIC: {modelo_sm_selected.aic:.2f}") # AIC from statsmodels OLS with selected features
print(f"BIC: {modelo_sm_selected.bic:.2f}") # BIC from statsmodels OLS with selected features

# Metrics for OLS model without jobclass_industrial and region_south - from cell 1bqPQl5zjcli
if 'modelo_sm_reduced_3' in globals():
    X_test_reduced_const_3 = sm.add_constant(X_test[X_train_reduced_3.columns])
    y_pred_reduced_3 = modelo_sm_reduced_3.predict(X_test_reduced_const_3)
    rmse_reduced_3 = np.sqrt(mean_squared_error(y_test, y_pred_reduced_3))
    mae_reduced_3 = mean_absolute_error(y_test, y_pred_reduced_3)
    r2_reduced_3 = r2_score(y_test, y_pred_reduced_3)
    n_reduced_3 = X_test[X_train_reduced_3.columns].shape[0]
    p_reduced_3 = X_test[X_train_reduced_3.columns].shape[1]
    adj_r2_reduced_3 = 1 - (1 - r2_reduced_3) * (n_reduced_3 - 1) / (n_reduced_3 - p_reduced_3 - 1)
    mape_reduced_3 = np.mean(np.abs((y_test - y_pred_reduced_3) / (y_test + epsilon))) * 100


    print("\nOLS Model (without jobclass_industrial and region_south):")
    print(f"R2: {r2_reduced_3:.4f}")
    print(f"Adjusted R2: {adj_r2_reduced_3:.4f}")
    print(f"RMSE: {rmse_reduced_3:.2f}")
    print(f"MAE: {mae_reduced_3:.2f}")
    print(f"MAPE: {mape_reduced_3:.2f}%")
    print(f"AIC: {modelo_sm_reduced_3.aic:.2f}")
    print(f"BIC: {modelo_sm_reduced_3.bic:.2f}")
else:
    print("\nOLS Model (without jobclass_industrial and region_south): Metrics not available. Please run the corresponding model fitting cell.")

# Metrics for OLS model without jobclass_information and region_east - from cell ArY-MKqXneeg
if 'modelo_sm_reduced_4' in globals():
    X_test_reduced_const_4 = sm.add_constant(X_test[X_train_reduced_4.columns])
    y_pred_reduced_4 = modelo_sm_reduced_4.predict(X_test_reduced_const_4)
    rmse_reduced_4 = np.sqrt(mean_squared_error(y_test, y_pred_reduced_4))
    mae_reduced_4 = mean_absolute_error(y_test, y_pred_reduced_4)
    r2_reduced_4 = r2_score(y_test, y_pred_reduced_4)
    n_reduced_4 = X_test[X_train_reduced_4.columns].shape[0]
    p_reduced_4 = X_test[X_train_reduced_4.columns].shape[1]
    adj_r2_reduced_4 = 1 - (1 - r2_reduced_4) * (n_reduced_4 - 1) / (n_reduced_4 - p_reduced_4 - 1)
    mape_reduced_4 = np.mean(np.abs((y_test - y_pred_reduced_4) / (y_test + epsilon))) * 100

    print("\nOLS Model (without jobclass_information and region_east):")
    print(f"R2: {r2_reduced_4:.4f}")
    print(f"Adjusted R2: {adj_r2_reduced_4:.4f}")
    print(f"RMSE: {rmse_reduced_4:.2f}")
    print(f"MAE: {mae_reduced_4:.2f}")
    print(f"MAPE: {mape_reduced_4:.2f}%")
    print(f"AIC: {modelo_sm_reduced_4.aic:.2f}")
    print(f"BIC: {modelo_sm_reduced_4.bic:.2f}")
else:
    print("\nOLS Model (without jobclass_information and region_east): Metrics not available. Please run the corresponding model fitting cell.")


# Metrics for Ridge and Lasso (CV) - from cell gu0tmclYgBJy
# Ensure cell gu0tmclYgBJy has been run to have these variables and models defined
# Ridge
if 'best_ridge_model' in globals(): # Check if CV and model training were performed
    aic_ridge, bic_ridge = calculate_aic_bic(best_ridge_model, X_train_regularized, y_train)
    print("\nRidge Regression (with Cross-Validation):")
    print(f"Best alpha: {best_alpha_ridge:.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge_cv)):.2f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred_ridge_cv):.2f}")
    mape_ridge_cv = np.mean(np.abs((y_test - y_pred_ridge_cv) / (y_test + epsilon))) * 100
    print(f"MAPE: {mape_ridge_cv:.2f}%")
    print(f"R2: {r2_ridge_cv:.4f}")
    print(f"AIC (Approx): {aic_ridge:.2f}")
    print(f"BIC (Approx): {bic_ridge:.2f}")

else:
    print("\nRidge Regression (with Cross-Validation): Metrics not available. Please run the cross-validation cell.")


# Lasso
if 'best_lasso_model' in globals(): # Check if CV and model training were performed
    aic_lasso, bic_lasso = calculate_aic_bic(best_lasso_model, X_train_regularized, y_train)
    print("\nLasso Regression (with Cross-Validation):")
    print(f"Best alpha: {best_alpha_lasso:.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso_cv)):.2f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred_lasso_cv):.2f}")
    mape_lasso_cv = np.mean(np.abs((y_test - y_pred_lasso_cv) / (y_test + epsilon))) * 100
    print(f"MAPE: {mape_lasso_cv:.2f}%")
    print(f"R2: {r2_lasso_cv:.4f}")
    print(f"AIC (Approx): {aic_lasso:.2f}")
    print(f"BIC (Approx): {bic_lasso:.2f}")
else:
     print("\nLasso Regression (with Cross-Validation): Metrics not available. Please run the cross-validation cell.")

--- Model Performance Summary ---

Linear Regression (OLS - All Features):
R2: 0.0030
Adjusted R2: 0.0002
RMSE: 15271.20
MAE: 12233.81
MAPE: 30.64%
AIC: 46363.59
BIC: 46403.13

Backward Elimination Model:
R-squared (R²): 0.0015
Root Mean Squared Error (RMSE): 15274.06
Mean Absolute Error (MAE): 12221.75
AIC: 46361.06
BIC: 46389.31

OLS Model (without jobclass_industrial and region_south):
R2: 0.0019
Adjusted R2: -0.0048
RMSE: 15271.20
MAE: 12233.81
MAPE: 30.64%
AIC: 46363.59
BIC: 46403.13

OLS Model (without jobclass_information and region_east):
R2: 0.0019
Adjusted R2: -0.0048
RMSE: 15271.20
MAE: 12233.81
MAPE: 30.64%
AIC: 46363.59
BIC: 46403.13

Ridge Regression (with Cross-Validation):
Best alpha: 10000.0000
RMSE: 15291.19
MAE: 12245.78
MAPE: 30.69%
R2: -0.0008
AIC (Approx): 40411.91
BIC (Approx): 40462.75

Lasso Regression (with Cross-Validation):
Best alpha: 291.5053
RMSE: 15292.98
MAE: 12246.52
MAPE: 30.69%
R2: -0.0010
AIC (Approx): 40412.06
BIC (Approx): 40462.91


In [182]:
print("\n--- Ridge Regression Coefficients ---")
for feature, coef in zip(X_train_regularized.columns, ridge_model.coef_):
    print(f"{feature}: {coef:.4f}")

print("\n--- Lasso Regression Coefficients ---")
for feature, coef in zip(X_train_regularized.columns, lasso_model.coef_):
    print(f"{feature}: {coef:.4f}")


--- Ridge Regression Coefficients ---
age: -49.6373
experience: 35.1027
education_encoded: -16.7738
jobclass_Industrial: 63.0750
jobclass_Information: -63.0750
region_East: -24.1718
region_South: -420.3598
region_West: 444.5316

--- Lasso Regression Coefficients ---
age: -49.6245
experience: 35.0893
education_encoded: -16.0712
jobclass_Industrial: 121.9892
jobclass_Information: -0.0000
region_East: -0.0000
region_South: -393.6824
region_West: 466.1931


### Análisis

Comparando las métricas de desempeño de algunos de los modelos realizados, se encontró que la regresión regularizada no mejoró el desempeño de los modelos anteriores, al mismo tiempo se descubrió que el modelo con el mejor desempeño encontrado fue el original con todas las variables.

Se probaron modelos en la base de entrenamiento con desempeño similar al modelo original pero que al probarlos en la base de prueba, el rendimiento disminuyó.

Se encontró multicolinealidad en las variables age y experience, se realizaron dos modelos ols eliminando una de las dos variables, pero el desempeñó general se desplomó en ambas ocasiones.

Posteriormente, por medio de la construcción de diversos modelos por prueba y error se descubrió que la multicolinealidad desaparecía al eliminar las variables region y jobclass, sin embargo hacerlo, disminuía en gran medida la capacidad de predicción del modelo.

# Informe ejecutivo

## Hallazgos principales:
Las variables que más impactaron los salarios en base a los modelos realizados fueron edad y experiencia.


Al realizar los modelos regularizados no se encontró ninguna mejora con respecto a los modelos clásicos

## Implicaciones para la gestión del talento humano:

La baja capacidad explicativa de los modelos implica que los factores considerados no son suficientes para entender la estructura salarial.

Ignorar problemas estadísticos como la multicolinealidad puede llevar a:
*  Interpretaciones erróneas sobre qué variables impactan realmente los salarios.
*  Políticas de compensación ineficientes, al basarse en información inestable o sesgada.

## Recomendaciones:

Recolección de mejores datos: Incluir variables adicionales relevantes para el salario (educación, experiencia, desempeño, industria, género, ubicación).

Uso de modelos más robustos: Complementar la regresión lineal con técnicas no lineales (árboles de decisión, Random Forest, XGBoost) que capturen interacciones y relaciones complejas.

Gestión del riesgo estadístico: Aplicar técnicas de regularización (Ridge) como práctica estándar para mitigar multicolinealidad y mejorar estabilidad en las estimaciones.

Política salarial basada en evidencia: Priorizar la identificación de factores que realmente expliquen las diferencias salariales, para diseñar esquemas de compensación más justos y competitivos.